# Zadanie 5: programowanie genetyczne i regresja symboliczna

Adam Tokarz, Serhii Zeliuk

EAIIIB ISI Metody i algorytmy optymalizacji grupa laboratoryjna 2 poniedziałek 18:30

Termin realizacji: 12 maja 2025

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zmodyfikuj przykład `pysr_demo.ipynb` tak, aby uczył się funkcji $f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$ której dziedziną jest $\mathbb{R}^6$. Uczenie ma się odbywać w oparciu o 200 wylosowanych z dziedziny próbek (między -5 a 5).
2. Zanotuj wzory trzech rozwiązań o najwyższej wartości `score` oraz rozwiązanie `best` dla następujących zestawów ustawień:
   1. `binary_operators=["+", "*"], unary_operators=["cos", "exp", "sin"], maxsize=20`,
   2. `binary_operators=["+", "*", "-", "^"], unary_operators=["cos", "exp", "sin", "log"], maxsize=30`, (dodaj ograniczenie dla argumentów operatora "^").
   3. `binary_operators=["+", "*", "-", "^"], unary_operators=["exp", "sin"], maxsize=15`.
3. Powtórz eksperymenty z zadania na 3.0 po dodaniu szumu do próbek z funkcji $f$ (rozkład normalny o średniej 0 i odchyleniu standardowym 0.5)

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj do porównania dopasowanie oparte o próbki losowane w szerszym zakresie (między -15 a 15) oraz wyższy poziom szumu (odchylenie standardowe równe 2 oraz 5).

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zamień funkcję $f$ na $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$ gdzie $p(i)$ oznacza $i$-tą liczbę pierwszą. Uwzględnij `p` jako dodatkowy operator unarny analogicznie do przykładu "Julia packages and types" z notatnika `pysr_demo.ipynb`. Powtórz eksperymenty opisane w zadaniach na 3.0 i 4.0.


In [101]:
import pysr
import sympy
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split

# 5.1 Function f [3.0]

$$f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$$

- X (random values  $\mathbb{R}^6$ [-5, 5])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev = 0.5) 

In [102]:
from numpy import sin, cos, exp, log, sqrt
# Dataset

np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
# y = 2.5382 * np.cos(X[:, 3]) + X[:, 0] ** 2 - 2
y = 2.2*sin(X[:, 0] + 2*X[:, 1]) - X[:, 5] ** 2 - 3
# Add the y samples with random noise
noise = np.random.normal(loc=0.0, scale=0.5, size=y.shape)
y_noisy = y + noise

In [103]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.2 Experiments [3.0]



## 5.2.1 Model 1

In [104]:
model_1 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.159
3           5.546e+01  1.240e-03  y = x₂ + -11.311
4           5.152e+01  7.370e-02  y = cos(x₅) + -10.987
5           6.759e+00  2.031e+00  y = (x₅ * x₅) * -1.1931
7           2.205e+00  5.600e-01  y = ((x₅ * x₅) + 3.2226) * -0.98154
9           2.197e+00  1.884e-03  y = (((x₅ + 0.033225) * -0.9792) * x₅) + -3.1779
11          2.173e+00  5.516e-03  y = (((x₅ * x₅) + (x₀ * -0.060343)) + 3.2193) * -0.98037
12          2.108e+00  3.018e-02  y = (sin(x₀) * -0.4006) + (((x₅ * x₅) + 3.2123) * -0.98316...
                                      )
13          2.096e+00  5.928e-03  y = ((cos(sin(x₁ + 2.3383)) + (x₅ * x₅)) + 2.4617) * -0.98...
                                      245
14          1.969e+00  6.239e-02  y = ((x₅ * x₅) + (cos(sin(cos(x₀) + x₁)) + 2.4525)) * -0.9...
                                

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -11.158837   
	1         0.001240                                    x2 + -11.310887   
	2         0.073704                               cos(x5) + -10.987488   
	3         2.031154                             (x5 * x5) * -1.1931033   
	4   >>>>  0.560019              ((x5 * x5) + 3.2226162) * -0.98154366   
	5         0.001884  (((x5 + 0.03322506) * -0.97919613) * x5) + -3....   
	6         0.005516  (((x5 * x5) + (x0 * -0.060342833)) + 3.2192686...   
	7         0.030184  (sin(x0) * -0.400605) + (((x5 * x5) + 3.212305...   
	8         0.005928  ((cos(sin(x1 + 2.3382633)) + (x5 * x5)) + 2.46...   
	9         0.062391  ((x5 * x5) + (cos(sin(cos(x0) + x1)) + 2.45252...   
	10        0.001407  (((x5 * x5) + cos(sin(x1 + sin(cos(x0))))) + 2...   
	11        0.183691  ((x5 * x5) + (cos(cos(x1 * 2.0104563) + sin(x0...   
	
	         loss  complexity  
	0   55.599790           1  
	1   55.462025           3  
	2   51.521240           4  
	3    6.758767           5  
	4    2.205166           7  
	5    2.196871           9  
	6    2.172767          11  
	7    2.108165          12  
	8    2.095705          13  
	9    1.968948          14  
	10   1.966179          15  
	11   1.636239          16  
]

  - outputs\20250511_222115_uo030Y\hall_of_fame.csv


### Model 1 best + top3 according to score

In [105]:
best_equation_1 = model_1.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: (x5*x5 + 3.2226162)*(-0.98154366)
Index 3, score=2.031154
Equation: (x5 * x5) * -1.1931033

Index 4, score=0.560019
Equation: ((x5 * x5) + 3.2226162) * -0.98154366

Index 11, score=0.183691
Equation: ((x5 * x5) + (cos(cos(x1 * 2.0104563) + sin(x0)) + 2.4525237)) * -0.98479724



## 5.2.2 Model 2

Added constraint to operator "^" - safe_pow

In [106]:
%%julia
function safe_pow(x::T, y::T) where T
    if(x < 0 && floor(y) != y)
        return T(NaN)
    else
        return T(x^y)
    end
end

safe_pow (generic function with 2 methods)

In [107]:
def safe_pow(x, y):
    if x < 0 and np.floor(y) != y:
        return np.nan
    return np.power(x, y)

model_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.158
3           5.537e+01  2.051e-03  y = -11.021 - x₅
4           5.152e+01  7.208e-02  y = cos(x₅) + -10.987
5           2.224e+00  3.143e+00  y = -3.0128 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = (-3.222 - (x₅ * x₅)) * 0.98156
9           2.153e+00  1.189e-02  y = (((x₄ * 0.03329) - x₅) * x₅) - 3.0191
10          2.134e+00  9.020e-03  y = ((sin(x₀) * -0.41058) - (x₅ * x₅)) + -3.0123
11          2.113e+00  1.002e-02  y = (-2.2633 - (x₅ * x₅)) - cos(sin(-2.3503 - x₁))
12          2.106e+00  3.265e-03  y = (((-3.2719 - (x₅ * x₅)) * 2.2499) - sin(x₀)) * 0.43434
13          2.082e+00  1.132e-02  y = (-2.2528 - (x₅ * x₅)) - cos(-0.2715 - sin(x₁ - -2.2828...
                                      ))
14          2.021e+00  2.991e-02  y = (-2.2551 - (x₅ * x₅)) - cos(sin(x₁ - (cos(x₀) + 10.991...
       

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -11.158208   
	1         0.002051                                    -11.020786 - x5   
	2         0.072084                               cos(x5) + -10.987481   
	3   >>>>  3.142667                              -3.012808 - (x5 * x5)   
	4         0.004262                (-3.222021 - (x5 * x5)) * 0.9815646   
	5         0.011890        (((x4 * 0.03329027) - x5) * x5) - 3.0190992   
	6         0.009020   ((sin(x0) * -0.4105781) - (x5 * x5)) + -3.012345   
	7         0.010018  (-2.2633302 - (x5 * x5)) - cos(sin(-2.3503397 ...   
	8         0.003265  (((-3.2719142 - (x5 * x5)) * 2.2499301) - sin(...   
	9         0.011323  (-2.252766 - (x5 * x5)) - cos(-0.27150393 - si...   
	10        0.029914  (-2.255101 - (x5 * x5)) - cos(sin(x1 - (cos(x0...   
	11        0.035602  (-2.228514 - (x5 * x5)) - cos(sin(0.8626206 - ...   
	12        0.001094  (-2.214954 - (x5 * x5)) - cos(-0.10120908 - si...   
	
	         loss  complexity  
	0   55.599790           1  
	1   55.372204           3  
	2   51.521240           4  
	3    2.224043           5  
	4    2.205166           7  
	5    2.153344           9  
	6    2.134008          10  
	7    2.112735          11  
	8    2.105849          12  
	9    2.082138          13  
	10   2.020775          14  
	11   1.881890          16  
	12   1.867531          23  
]

In [108]:
best_equation_2 = model_2.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 3.012808
Index 3, score=3.142667
Equation: -3.012808 - (x5 * x5)

Index 2, score=0.072084
Equation: cos(x5) + -10.987481

Index 11, score=0.035602
Equation: (-2.228514 - (x5 * x5)) - cos(sin(0.8626206 - (x1 - cos(x0 + 11.154672))))



## 5.2.3 Model 3

In [109]:
model_3 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.159
3           5.537e+01  2.051e-03  y = -11.021 - x₅
4           5.462e+01  1.364e-02  y = -11.157 - sin(x₀)
5           2.224e+00  3.201e+00  y = -3.0128 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = ((x₅ * -0.98155) * x₅) - 3.1631
9           2.153e+00  1.189e-02  y = -3.0191 - (x₅ * (x₅ + (x₄ * -0.03329)))
10          2.134e+00  9.020e-03  y = -3.0123 - ((sin(x₀) * 0.41058) + (x₅ * x₅))
11          2.126e+00  3.635e-03  y = ((exp(sin(x₀)) * -0.37145) - (x₅ * x₅)) + -2.536
12          2.122e+00  1.945e-03  y = ((sin(x₀ * -1.1376) * 0.43486) - (x₅ * x₅)) + -2.9953
───────────────────────────────────────────────────────────────────────────────────────────────────


PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                         -11.158751   
	1        0.002051                                    -11.020784 - x5   
	2        0.013642                               -11.157398 - sin(x0)   
	3  >>>>  3.201109                             -3.0128005 - (x5 * x5)   
	4        0.004262              ((x5 * -0.98154527) * x5) - 3.1631148   
	5        0.011890     -3.0190911 - (x5 * (x5 + (x4 * -0.033289663)))   
	6        0.009021  -3.0123394 - ((sin(x0) * 0.41058457) + (x5 * x5))   
	7        0.003635  ((exp(sin(x0)) * -0.37145144) - (x5 * x5)) + -...   
	8        0.001945  ((sin(x0 * -1.1375549) * 0.43485788) - (x5 * x...   
	
	        loss  complexity  
	0  55.599790           1  
	1  55.372204           3  
	2  54.621952           4  
	3   2.224043           5  
	4   2.205166           7  
	5   2.153344           9  
	6   2.134007          10  
	7   2.126264          11  
	8   2.122133          12  
]

  - outputs\20250511_222121_LotbFM\hall_of_fame.csv


In [110]:
best_equation_3 = model_3.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 3.0128005
Index 3, score=3.201109
Equation: -3.0128005 - (x5 * x5)

Index 2, score=0.013642
Equation: -11.157398 - sin(x0)

Index 5, score=0.011890
Equation: -3.0190911 - (x5 * (x5 + (x4 * -0.033289663)))



## 5.2.4 Model 1 with noise

In [111]:
model_1_noisy.fit(X, y_noisy)
best_equation_1_noisy = model_1_noisy.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1_noisy.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.093
3           5.560e+01  5.027e-04  y = x₂ + -11.246
4           5.159e+01  7.493e-02  y = cos(x₅) + -10.922
5           6.855e+00  2.018e+00  y = x₅ * (x₅ * -1.188)
7           2.449e+00  5.148e-01  y = ((x₅ * -0.97982) * x₅) + -3.1117
9           2.434e+00  2.937e-03  y = (((x₅ * -0.97674) + -0.042769) * x₅) + -3.1309
11          2.415e+00  3.898e-03  y = (x₀ * 0.059985) + (((x₅ * -0.97861) * x₅) + -3.1043)
12          2.340e+00  3.182e-02  y = (sin(x₁ * -2.0207) + ((x₅ * -0.98091) * x₅)) + -3.1691
13          2.297e+00  1.843e-02  y = sin(sin(x₁ * -2.0156)) + ((x₅ * (x₅ * -0.98063)) + -3....
                                      1643)
14          2.264e+00  1.427e-02  y = ((x₅ * (x₅ * -0.98038)) + (sin(x₁ * -2.0074) * 0.60904...
                                      )) + -3.1492
15          1.486e+0

## 5.2.5 Model 2 with noise

In [112]:
model_2_noisy.fit(X, y_noisy)
best_equation_2_noisy = model_2_noisy.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2_noisy.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.093
3           5.528e+01  3.374e-03  y = -10.955 - x₅
4           5.159e+01  6.919e-02  y = cos(x₅) - 10.922
5           2.471e+00  3.039e+00  y = -2.9473 - (x₅ * x₅)
7           2.449e+00  4.586e-03  y = (-3.1757 - (x₅ * x₅)) * 0.97982
9           2.434e+00  2.937e-03  y = ((x₅ * -0.97674) * (x₅ - -0.043786)) - 3.1309
10          2.360e+00  3.105e-02  y = (sin(x₁ * -2.0251) + -3.0129) - (x₅ * x₅)
11          2.318e+00  1.792e-02  y = (sin(sin(x₁ * -2.0249)) + -3.0129) - (x₅ * x₅)
12          2.295e+00  9.896e-03  y = sin(cos(x₁) - sin(x₀)) + (-2.8286 - (x₅ * x₅))
14          1.900e+00  9.445e-02  y = sin((cos(x₁) - sin(x₀)) * 1.9893) + (-2.8404 - (x₅ * x...
                                      ₅))
16          1.891e+00  2.317e-03  y = (sin(((cos(x₁) - sin(x₀)) * 1.9739) - 0.11525) + -2.82...
         

## 5.2.6 Model 3 with noise

In [113]:
model_3_noisy.fit(X, y_noisy)
best_equation_3_noisy = model_3_noisy.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3_noisy.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.093
3           5.528e+01  3.374e-03  y = -10.956 - x₅
4           5.467e+01  1.109e-02  y = -11.092 - sin(x₀)
5           2.471e+00  3.097e+00  y = -2.9476 - (x₅ * x₅)
7           2.449e+00  4.586e-03  y = -3.1117 - ((x₅ * x₅) * 0.97982)
9           2.434e+00  2.937e-03  y = ((x₅ + 0.04379) * (x₅ * -0.97674)) + -3.1309
10          2.374e+00  2.486e-02  y = -1.8988 - ((x₅ * x₅) + safe_pow(1.525, sin(x₀)))
11          2.373e+00  4.920e-04  y = -1.5328 - (safe_pow(1.2868, exp(sin(x₀))) + (x₅ * x₅))
12          2.263e+00  4.738e-02  y = (-4.0896 - (x₅ * x₅)) + safe_pow(0.53521, sin(x₁ * 2.0...
                                      282))
14          2.263e+00  1.457e-05  y = (-4.0872 - (x₅ * x₅)) + safe_pow(0.53576, sin((x₁ * 2....
                                      0262) + -0.017702))
───────────────────

# 5.3 Output into a DF [3.0]

In [114]:
import pandas as pd

data = []

for model_name in ['model_1', 'model_2', 'model_3','model_1_noisy', 'model_2_noisy', 'model_3_noisy']:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,(x5*x5 + 3.2226162)*(-0.98154366),2.031154
1,model_1,3,(x5 * x5) * -1.1931033,2.031154
2,model_1,4,((x5 * x5) + 3.2226162) * -0.98154366,0.560019
3,model_1,11,((x5 * x5) + (cos(cos(x1 * 2.0104563) + sin(x0...,0.183691
4,model_2,best,-x5*x5 - 3.012808,3.142667
5,model_2,3,-3.012808 - (x5 * x5),3.142667
6,model_2,2,cos(x5) + -10.987481,0.072084
7,model_2,11,(-2.228514 - (x5 * x5)) - cos(sin(0.8626206 - ...,0.035602
8,model_3,best,-x5*x5 - 3.0128005,3.201109
9,model_3,3,-3.0128005 - (x5 * x5),3.201109


In [115]:
df.to_csv('results3.csv')

# 5.4 Function f [4.0]

$$f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$$

- X (random values  $\mathbb{R}^6$ [-15, 15])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev1 = 2, std_dev2 = 5) 

In [116]:
from numpy import sin, cos, exp, log, sqrt
# Dataset

np.random.seed(0)
X = np.random.uniform(-15, 15, size=(200, 6))
# y = 2.5382 * np.cos(X[:, 3]) + X[:, 0] ** 2 - 2
y = 2.2*sin(X[:, 0] + 2*X[:, 1]) - X[:, 5] ** 2 - 3
# Add the y samples with random noise
noise1 = np.random.normal(loc=0.0, scale=2, size=y.shape)
y_noisy1 = y + noise1
noise2 = np.random.normal(loc=0.0, scale=5, size=y.shape)
y_noisy2 = y + noise2

In [117]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.5 Experiments [4.0]



## 5.5.1 Model 1

In [118]:
model_1_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.289
3           4.338e+03  1.665e-02  y = x₂ + -76.745
5           6.228e+00  3.273e+00  y = (x₅ * x₅) * -1.0218
7           2.054e+00  5.546e-01  y = (x₅ * (x₅ * -0.99929)) + -3.0285
9           2.022e+00  7.920e-03  y = (x₅ * ((x₅ + -0.021406) * -0.99981)) + -2.9997
12          2.022e+00  1.832e-05  y = (x₅ * (x₅ * -0.99981)) + (sin(x₅ * 0.021771) + -2.9993...
                                      )
13          1.967e+00  2.765e-02  y = sin(exp(x₃) * -2.7684) + (-3.0624 + ((x₅ * x₅) * -0.99...
                                      904))
15          1.936e+00  7.748e-03  y = sin(exp(x₃) * -2.7684) + (-3.0624 + ((x₅ + -0.030585) ...
                                      * (x₅ * -0.99904)))
16          1.927e+00  4.748e-03  y = (x₅ * ((x₅ + -0.02574) * -0.9998)) + ((sin(x₀ * -1.352...
                    

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          -76.28887   
	1         0.016651                                     x2 + -76.74527   
	2         3.273026                             (x5 * x5) * -1.0218015   
	3   >>>>  0.554636               (x5 * (x5 * -0.9992922)) + -3.028511   
	4         0.007920  (x5 * ((x5 + -0.021405697) * -0.99980783)) + -...   
	5         0.000018  (x5 * (x5 * -0.9998088)) + (sin(x5 * 0.0217710...   
	6         0.027652  sin(exp(x3) * -2.7683747) + (-3.0624006 + ((x5...   
	7         0.007748  sin(exp(x3) * -2.7683747) + (-3.0624006 + ((x5...   
	8         0.004748  (x5 * ((x5 + -0.025740454) * -0.9997968)) + ((...   
	9         0.006598  sin(((x5 + -0.021124912) * -0.99983114) * exp(...   
	10        0.004016  -3.0005355 + (sin(sin(exp(x3) * (-0.99983114 *...   
	11        0.026921  -3.0005355 + (sin(exp(x3) * (-0.99983114 * (x5...   
	12        0.003147  (-3.0005355 + sin(sin(exp(x3) * (-0.99983114 *...   
	
	           loss  complexity  
	0   4484.837400           1  
	1   4337.941400           3  
	2      6.228301           5  
	3      2.054086           7  
	4      2.021806           9  
	5      2.021694          12  
	6      1.966556          13  
	7      1.936319          15  
	8      1.927146          16  
	9      1.914473          17  
	10     1.906800          18  
	11     1.856152          19  
	12     1.850321          20  
]

### Model 1 best + top3 according to score

In [119]:
best_equation_1 = model_1_wide_range.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x5*x5*(-0.9992922) - 3.028511
Index 2, score=3.273026
Equation: (x5 * x5) * -1.0218015

Index 3, score=0.554636
Equation: (x5 * (x5 * -0.9992922)) + -3.028511

Index 6, score=0.027652
Equation: sin(exp(x3) * -2.7683747) + (-3.0624006 + ((x5 * x5) * -0.9990412))



## 5.5.2 Model 2

In [120]:

model_2_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.289
3           4.338e+03  1.665e-02  y = x₂ - 76.744
5           2.056e+00  3.827e+00  y = -2.9766 - (x₅ * x₅)
7           2.022e+00  8.427e-03  y = ((0.021657 - x₅) * x₅) + -2.9856
9           2.022e+00  4.020e-05  y = ((0.021379 - x₅) * (x₅ * 0.99981)) - 2.9996
11          2.022e+00  2.384e-07  y = (x₅ + -2.9996) - (((x₅ + 0.9788) * 0.99981) * x₅)
12          1.989e+00  1.653e-02  y = (x₅ - ((cos(x₄ * -0.02477) + x₅) * x₅)) + -2.9894
14          1.987e+00  4.635e-04  y = (x₅ + -2.9881) - (x₅ * (cos((x₄ + -0.74795) * -0.02581...
                                      5) + x₅))
16          1.986e+00  1.711e-04  y = x₅ + ((-4.0952 - ((cos((x₄ + -1.1541) * -0.024746) + x...
                                      ₅) * x₅)) + 1.1059)
17          1.979e+00  3.846e-03  y = x₅ + (-3.3427 - (((cos(-0.024545 * (x₄

PySRRegressor.equations_ = [
	    pick         score                                           equation  \
	0         0.000000e+00                                          -76.28853   
	1         1.665117e-02                                      x2 - 76.74393   
	2   >>>>  3.827115e+00                             -2.9765701 - (x5 * x5)   
	3         8.426933e-03             ((0.021656781 - x5) * x5) + -2.9856153   
	4         4.025939e-05  ((0.021379333 - x5) * (x5 * 0.99980634)) - 2.9...   
	5         2.967643e-07  (x5 + -2.9996362) - (((x5 + 0.9787984) * 0.999...   
	6         1.653516e-02  (x5 - ((cos(x4 * -0.024769949) + x5) * x5)) + ...   
	7         4.635442e-04  (x5 + -2.9880662) - (x5 * (cos((x4 + -0.747945...   
	8         1.711581e-04  x5 + ((-4.0952125 - ((cos((x4 + -1.1541268) * ...   
	9         3.846013e-03  x5 + (-3.3426971 - (((cos(-0.024544515 * (x4 +...   
	10        5.297801e-03  ((x4 * 0.016947726) + -1.0584998) + (x5 + (-1....   
	11        9.047590e-04  ((-1.4729519 - ((x4 * -0.017797565) + ((cos((x...   
	12        1.396603e-03  x5 + (-3.0087636 - (((cos(cos(sin(exp(x0 + -0....   
	13        1.347626e-02  (x5 + -2.9646902) - (((cos(cos(sin((x4 + cos(x...   
	
	           loss  complexity  
	0   4484.837400           1  
	1   4337.941400           3  
	2      2.056337           5  
	3      2.021970           7  
	4      2.021807           9  
	5      2.021806          11  
	6      1.988650          12  
	7      1.986807          14  
	8      1.986127          16  
	9      1.978503          17  
	10     1.968049          18  
	11     1.964491          20  
	12     1.961749          21  
	13     1.909581          23  
]

  - outputs\20250511_222133_GnbNXP\hall_of_fame.csv


In [121]:
best_equation_2 = model_2_wide_range.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 2.9765701
Index 2, score=3.827115
Equation: -2.9765701 - (x5 * x5)

Index 1, score=0.016651
Equation: x2 - 76.74393

Index 6, score=0.016535
Equation: (x5 - ((cos(x4 * -0.024769949) + x5) * x5)) + -2.9894004



## 5.5.3 Model 3

In [122]:
model_3_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.29
3           4.338e+03  1.665e-02  y = x₂ + -76.745
5           2.056e+00  3.827e+00  y = -2.9766 - (x₅ * x₅)
7           2.022e+00  8.427e-03  y = ((0.021651 - x₅) * x₅) + -2.9856
9           2.022e+00  4.044e-05  y = ((x₅ * -0.99981) * (x₅ + -0.021392)) - 2.9996
11          2.022e+00  2.980e-07  y = (((x₅ * -0.99981) + -0.97861) * x₅) + (x₅ + -2.9996)
12          2.013e+00  4.236e-03  y = (x₅ * (((sin(x₃) * -0.033706) + x₅) * -0.99966)) + -3....
                                      0326
13          2.003e+00  5.049e-03  y = ((x₅ * (x₅ * -0.99836)) + sin(exp(x₅) * -0.48261)) + -...
                                      2.985
14          1.984e+00  9.512e-03  y = ((x₅ + (sin(x₃) * -0.032386)) * ((x₅ * -1.0001) + 0.02...
                                      0341)) + -3.0049
15          1.927e+00  2.92

PySRRegressor.equations_ = [
	   pick         score                                           equation  \
	0        0.000000e+00                                         -76.290085   
	1        1.665117e-02                                     x2 + -76.74524   
	2  >>>>  3.827114e+00                              -2.976615 - (x5 * x5)   
	3        8.426882e-03             ((0.021650687 - x5) * x5) + -2.9855635   
	4        4.050669e-05  ((x5 * -0.99980634) * (x5 + -0.021392154)) - 2...   
	5        3.709555e-07  (((x5 * -0.99980634) + -0.97860956) * x5) + (x...   
	6        4.236272e-03  (x5 * (((sin(x3) * -0.033706352) + x5) * -0.99...   
	7        5.049288e-03  ((x5 * (x5 * -0.9983649)) + sin(exp(x5) * -0.4...   
	8        9.512338e-03  ((x5 + (sin(x3) * -0.03238588)) * ((x5 * -1.00...   
	9        2.920779e-02  (((x5 * x5) * -1.0002514) + (sin(5.593277 - ex...   
	
	          loss  complexity  
	0  4484.837400           1  
	1  4337.941400           3  
	2     2.056337           5  
	3     2.021970           7  
	4     2.021806           9  
	5     2.021805          11  
	6     2.013258          12  
	7     2.003118          13  
	8     1.984154          14  
	9     1.927040          15  
]

In [123]:
best_equation_3 = model_3_wide_range.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 2.976615
Index 2, score=3.827114
Equation: -2.976615 - (x5 * x5)

Index 9, score=0.029208
Equation: (((x5 * x5) * -1.0002514) + (sin(5.593277 - exp(x5)) * 0.64666533)) + -2.7202268

Index 1, score=0.016651
Equation: x2 + -76.74524



## 5.5.4 Model 1 with noise std_dev 2 and 5

In [124]:
model_1_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_1_noisy = model_1_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.024
3           4.336e+03  1.655e-02  y = x₂ + -76.484
5           9.261e+00  3.074e+00  y = x₅ * (x₅ * -1.0195)
7           5.635e+00  2.484e-01  y = ((x₅ * -0.99853) * x₅) + -2.8217
9           5.631e+00  3.749e-04  y = (x₅ * ((x₅ * -0.99871) + 0.0077384)) + -2.8122
11          5.541e+00  8.079e-03  y = ((sin(exp(x₄)) + (x₅ * x₅)) * -0.99876) + -2.7275
12          5.513e+00  5.021e-03  y = ((x₅ * x₅) * -0.99894) + (sin(x₅ * -0.91938) + -2.8934...
                                      )
13          5.411e+00  1.870e-02  y = ((sin(exp(x₅ + x₅)) + (x₅ * x₅)) * -0.99876) + -2.7275
14          5.220e+00  3.593e-02  y = ((sin(x₀ * -1.4575) + 104.2) * (x₅ * (x₅ * -0.0095822)...
                                      )) + -2.8465
16          5.168e+00  4.928e-03  y = (((sin((x₀ * -1.4619) + -0.34131) + 104.2) *

In [125]:
model_1_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_1_noisy = model_1_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.133
3           4.394e+03  1.726e-02  y = x₂ + -76.589
5           2.874e+01  2.515e+00  y = (x₅ * -1.0227) * x₅
7           2.578e+01  5.427e-02  y = (x₅ * (x₅ * -1.0037)) + -2.5489
9           2.573e+01  9.794e-04  y = ((x₅ + 0.026665) * (x₅ * -1.0031)) + -2.5848
10          2.532e+01  1.615e-02  y = ((x₅ * (x₅ * -1.0035)) + cos(x₄)) + -2.6166
11          2.495e+01  1.475e-02  y = ((x₅ * (x₅ * -1.003)) + -2.511) + (x₀ * 0.10008)
12          2.467e+01  1.131e-02  y = (((x₅ * x₅) * -1.004) + sin(x₄ * x₅)) + -2.5133
14          2.455e+01  2.485e-03  y = ((((x₅ * -1.0028) * x₅) + (x₀ * 0.096411)) + cos(x₄)) ...
                                      + -2.5799
15          2.429e+01  1.059e-02  y = (cos(x₄) + sin(x₃ * -9.8827)) + (((x₅ * x₅) * -1.0029)...
                                       + -2.6657)
16  

## 5.5.5 Model 2 with noise std_dev 2 and 5

In [126]:
model_2_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_2_noisy = model_2_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.022
3           4.336e+03  1.655e-02  y = x₂ - 76.484
5           5.645e+00  3.322e+00  y = -2.7147 - (x₅ * x₅)
7           5.635e+00  8.623e-04  y = (-2.8268 - (x₅ * x₅)) * 0.99853
9           5.547e+00  7.834e-03  y = (-2.6367 - (x₅ * x₅)) - sin(exp(x₄))
10          5.160e+00  7.235e-02  y = -2.7376 - ((x₅ * x₅) - sin(x₀ * 1.4556))
12          5.142e+00  1.791e-03  y = (-2.7376 - (x₅ * x₅)) - sin((4.1818 - x₀) * 1.4556)
14          5.142e+00  8.941e-08  y = -2.3927 - (((x₅ * x₅) - sin((6.3401 - x₀) * 1.4556)) -...
                                       -0.3449)
17          5.078e+00  4.153e-03  y = -2.7195 - ((x₅ * x₅) - sin((1.9615 - sin(x₄ * 2.7322))...
                                       - (x₁ * 1.575)))
19          4.994e+00  8.394e-03  y = -2.7011 - ((x₅ * x₅) - sin((((1.9649 - sin(x₄ * 2.7288.

In [127]:
model_2_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_2_noisy = model_2_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.127
3           4.394e+03  1.726e-02  y = x₂ + -76.589
5           2.585e+01  2.568e+00  y = -2.8207 - (x₅ * x₅)
7           2.578e+01  1.385e-03  y = -2.8077 - (x₅ * (x₅ - -0.031293))
8           2.538e+01  1.561e-02  y = cos(x₄) - ((x₅ * x₅) - -2.8703)
10          2.474e+01  1.266e-02  y = sin(x₄ * x₅) - ((x₅ * x₅) + 2.8034)
12          2.449e+01  5.132e-03  y = (cos((x₅ * -1.9543) - x₂) - (x₅ * x₅)) - 2.744
14          2.449e+01  4.930e-05  y = (cos((x₅ * -1.9527) - (x₂ - -0.054362)) - (x₅ * x₅)) -...
                                       2.7435
15          2.438e+01  4.330e-03  y = (cos((x₅ * -1.9527) - (x₂ - sin(x₁))) - (x₅ * x₅)) - 2...
                                      .7435
16          2.432e+01  2.507e-03  y = cos(((x₂ - -1.006) * (0.053245 - x₅)) * -1.611) - ((x₅...
                       

## 5.5.6 Model 3 with noise

In [128]:
model_3_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_3_noisy = model_3_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.022
3           4.336e+03  1.655e-02  y = x₂ + -76.488
5           5.645e+00  3.322e+00  y = -2.7146 - (x₅ * x₅)
7           5.635e+00  8.623e-04  y = (-2.8265 - (x₅ * x₅)) * 0.99853
9           5.553e+00  7.303e-03  y = (-2.7135 - (x₅ * x₅)) - sin(exp(x₄))
10          5.159e+00  7.366e-02  y = -2.7464 - ((x₅ * x₅) - sin(x₀ * 1.4504))
12          5.142e+00  1.669e-03  y = sin((-0.13464 - x₀) * -1.4557) + (-2.7376 - (x₅ * x₅))
14          5.138e+00  3.796e-04  y = sin((-0.12976 - x₀) * -1.4551) + (-2.741 - (x₅ * (x₅ -...
                                       0.0073348)))
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_222147_tULtv1\hall_of_fame.csv
Best by score: -x5*x5 - 2.7146323
Index 2, score=3.321948
Equation: -2.7146323 - (x5 

In [129]:
model_3_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_3_noisy = model_3_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.134
3           4.394e+03  1.726e-02  y = x₂ - 76.589
5           2.585e+01  2.568e+00  y = -2.8207 - (x₅ * x₅)
7           2.578e+01  1.385e-03  y = -2.8077 - (x₅ * (x₅ + 0.03127))
9           2.499e+01  1.541e-02  y = (-2.7326 - (x₅ * x₅)) - (x₀ * -0.10117)
10          2.474e+01  1.011e-02  y = sin(x₄ * x₅) - ((x₅ * x₅) - -2.8037)
12          2.455e+01  3.930e-03  y = (sin((x₀ * x₂) * -2.0234) - (x₅ * x₅)) - 2.8387
13          2.426e+01  1.192e-02  y = (sin(x₀ * exp(x₂ * -3.1654)) - (x₅ * x₅)) - 2.7071
15          2.413e+01  2.693e-03  y = (sin(-2.0261 - (exp(x₀) * 37.025)) - ((x₅ * x₅) - -2.2...
                                      273)) - 0.23777
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_222149_FEpkyr\hall_of_fame.csv
Bes

# 5.6 Output into a DF [4.0]

In [130]:
import pandas as pd

data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5'   
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,(x5*x5 + 3.2226162)*(-0.98154366),2.031154
1,model_1,3,(x5 * x5) * -1.1931033,2.031154
2,model_1,4,((x5 * x5) + 3.2226162) * -0.98154366,0.560019
3,model_1,11,((x5 * x5) + (cos(cos(x1 * 2.0104563) + sin(x0...,0.183691
4,model_2,best,-x5*x5 - 3.012808,3.142667
5,model_2,3,-3.012808 - (x5 * x5),3.142667
6,model_2,2,cos(x5) + -10.987481,0.072084
7,model_2,11,(-2.228514 - (x5 * x5)) - cos(sin(0.8626206 - ...,0.035602
8,model_3,best,-x5*x5 - 3.0128005,3.201109
9,model_3,3,-3.0128005 - (x5 * x5),3.201109


In [131]:
df.to_csv('results4.csv')

In [132]:
data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5'   
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,(x5*x5 + 3.2226162)*(-0.98154366),2.031154
1,model_2,best,-x5*x5 - 3.012808,3.142667
2,model_3,best,-x5*x5 - 3.0128005,3.201109
3,model_1_noisy,best,x5*(-0.9861936)*x5 + sin(x1*(-2.0073497) + sin...,2.018258
4,model_2_noisy,best,-x5*x5 - cos(sin(sin((x1 - sin(x0))*1.9730418)...,3.038596
5,model_3_noisy,best,-x5*x5 - 2.9476075,3.096697
6,model_1_wide_range,best,x5*x5*(-0.9992922) - 3.028511,3.273026
7,model_2_wide_range,best,-x5*x5 - 2.9765701,3.827115
8,model_3_wide_range,best,-x5*x5 - 2.976615,3.827114
9,model_1_noisy_std_dev_2,best,x5*(-0.99853265)*x5 - 2.821686,3.074411


In [133]:
df.to_csv('results4_only_best.csv')

In [134]:
from pysr import jl
jl.seval("using Primes: prime")

In [135]:
%julia using Primes: prime

In [136]:
jl.seval(
    """
function p(i::T) where T
    if 0.5 < i < 1000
        return T(prime(round(Int, i)))
    else
        return T(NaN)
    end
end
"""
)

p (generic function with 1 method)

In [137]:
%%julia
function p(i::T) where T
    if 0.5 < i < 1000
        return T(prime(round(Int, i)))
    else
        return T(0.0)
    end
end

p (generic function with 1 method)

# 5.7 Function f [5.0]

$$f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$$

- X (random values  $\mathbb{R}^6$ [-5, 5])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev = 0.5) 

In [138]:
import numpy as np
from pysr import PySRRegressor
from math import sin
import sympy

np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))


from sympy import primerange
primes_list = [0] + list(primerange(1, 1000))
primes_dict = {i: primes_list[i] if i < len(primes_list) else 0 for i in range(1000)}

y = [
    2.2 * sin(X[i, 0] + 2 * X[i, 1]) - X[i, 5] ** 2 - primes_dict[abs(int(X[i, 0]))]
    for i in range(200)
]

# Dodaj szum
noise = np.random.normal(loc=0.0, scale=0.5, size=200)
y_noisy = np.array(y) + noise

In [139]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.8 Experiments [5.0]



## 5.8.1 Model 1

In [140]:
import sympy
class sympy_p(sympy.Function):
    pass

model_1f = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    timeout_in_seconds=60,
    maxsize=20,
    **default_pysr_params,
)

model_1f.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.176e+01  1.594e+01  y = -11.903
3           6.118e+01  4.750e-03  y = x₂ + -12.057
4           5.765e+01  5.949e-02  y = cos(x₅) + -11.732
5           1.548e+01  1.315e+00  y = (x₅ * x₅) * -1.2425
7           8.495e+00  3.000e-01  y = ((x₅ * -0.98039) * x₅) + -3.9176
9           8.481e+00  8.454e-04  y = ((x₅ * (x₅ + 0.043774)) * -0.97731) + -3.9367
10          7.922e+00  6.817e-02  y = ((x₅ * x₅) * -0.9596) + (-3.142 + cos(x₀))
11          7.431e+00  6.399e-02  y = cos(x₀) + (p((x₅ * x₅) + 7.3479) * -0.2231)
12          4.014e+00  6.158e-01  y = (((x₅ * x₅) * -0.9596) + (x₀ * sin(x₀))) + -3.142
15          2.916e+00  1.065e-01  y = ((((x₅ * x₅) * -0.9596) + (x₀ * sin(x₀))) + -3.142) + ...
                                      cos(x₀)
──────────────────────────────────────────────────────────────────────────────────────────────────

PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                         -11.903401   
	1        0.004750                                     x2 + -12.05692   
	2        0.059494                               cos(x5) + -11.732317   
	3        1.314799                             (x5 * x5) * -1.2424526   
	4        0.300026               ((x5 * -0.9803854) * x5) + -3.917589   
	5        0.000846  ((x5 * (x5 + 0.043773502)) * -0.97730887) + -3...   
	6        0.068167  ((x5 * x5) * -0.9596014) + (-3.1420352 + cos(x0))   
	7        0.063986  cos(x0) + (p((x5 * x5) + 7.347882) * -0.22309607)   
	8  >>>>  0.615845  (((x5 * x5) * -0.9596014) + (x0 * sin(x0))) + ...   
	9        0.106549  ((((x5 * x5) * -0.9596014) + (x0 * sin(x0))) +...   
	
	        loss  complexity  
	0  61.764057           1  
	1  61.180126           3  
	2  57.646427           4  
	3  15.479691           5  
	4   8.494999           7  
	5   8.480646           9  
	6   7.921812          10  
	7   7.430804          11  
	8   4.014002          12  
	9   2.915789          15  
]

### Model 1 best + top3 according to score

In [141]:
best_equation_1 = model_1f.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1f.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x0*sin(x0) + x5*x5*(-0.9596014) - 3.1420352
Index 3, score=1.314799
Equation: (x5 * x5) * -1.2424526

Index 8, score=0.615845
Equation: (((x5 * x5) * -0.9596014) + (x0 * sin(x0))) + -3.1420352

Index 4, score=0.300026
Equation: ((x5 * -0.9803854) * x5) + -3.917589



## 5.8.2 Model 2

Added constraint to operator "^" - safe_pow

In [142]:
def safe_pow(x, y):
    if x < 0 and np.floor(y) != y:
        return np.nan
    return np.power(x, y)

model_2f = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.176e+01  1.594e+01  y = -11.904
3           6.118e+01  4.750e-03  y = x₂ - 12.055
4           4.339e+01  3.435e-01  y = -9.5185 - p(x₅)
5           8.516e+00  1.628e+00  y = -3.7577 - (x₅ * x₅)
7           8.496e+00  1.206e-03  y = -3.885 - safe_pow(x₅ * x₅, 0.99382)
8           5.584e+00  4.196e-01  y = (x₀ - p(x₀)) - (x₅ * x₅)
10          3.815e+00  1.905e-01  y = ((x₀ - p(x₀)) * 1.4422) - (x₅ * x₅)
12          3.440e+00  5.173e-02  y = ((x₀ - p(x₀ * 1.0839)) - (x₅ * x₅)) - 1.0839
13          3.433e+00  1.891e-03  y = (x₀ - p(x₀ * 1.0839)) - ((x₅ * x₅) - cos(-3.1615))
14          3.430e+00  9.315e-04  y = (x₀ - p((x₀ - 0.4072) * 1.2973)) - ((x₅ * x₅) - -0.856...
                                      32)
15          3.305e+00  3.722e-02  y = (x₀ - p(x₀ - -0.45969)) - ((x₅ * x₅) - cos(x₀ - -0.375...
                                

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -11.903521   
	1         0.004750                                     x2 - 12.055429   
	2         0.343521                                  -9.518542 - p(x5)   
	3         1.628317                              -3.757748 - (x5 * x5)   
	4         0.001206          -3.884976 - safe_pow(x5 * x5, 0.99381673)   
	5         0.419646                           (x0 - p(x0)) - (x5 * x5)   
	6         0.190525             ((x0 - p(x0)) * 1.4421711) - (x5 * x5)   
	7         0.051730  ((x0 - p(x0 * 1.0839033)) - (x5 * x5)) - 1.083...   
	8         0.001891  (x0 - p(x0 * 1.0839033)) - ((x5 * x5) - cos(-3...   
	9         0.000932  (x0 - p((x0 - 0.40720046) * 1.2972634)) - ((x5...   
	10        0.037222  (x0 - p(x0 - -0.45969033)) - ((x5 * x5) - cos(...   
	11        0.030749  (((x0 + 0.07900929) - p(x0 - -0.38057357)) * 1...   
	12        0.000457  (((x0 + 0.07900929) - p(x0 - -0.38057357)) * 1...   
	13        0.040471  ((x0 - p((x0 - log(1.4137571)) * 1.2040874)) -...   
	14        0.022614  ((x0 * 1.2155299) - p(x0 - -0.8133181)) - ((x5...   
	15  >>>>  0.077951  ((x0 - p(x0 - -0.4466726)) * 1.2834208) - ((x5...   
	16        0.071224  ((-0.25085652 - p(x0 - (x0 * 1.16379))) - ((x5...   
	
	         loss  complexity  
	0   61.764057           1  
	1   61.180126           3  
	2   43.393124           4  
	3    8.516313           5  
	4    8.495794           7  
	5    5.584111           8  
	6    3.814751          10  
	7    3.439804          12  
	8    3.433305          13  
	9    3.430108          14  
	10   3.304779          15  
	11   3.107664          17  
	12   3.104825          19  
	13   2.981679          20  
	14   2.915008          21  
	15   2.696411          22  
	16   2.511042          23  
]

In [143]:
best_equation_2 = model_2f.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2f.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: (x0 - sympy_p(x0 - 1*(-0.4466726)))*1.2834208 - (x5*x5 - cos((x0 - sympy_p(x0 - 1*1.3226477))*(-0.67865336)))
Index 3, score=1.628317
Equation: -3.757748 - (x5 * x5)

Index 5, score=0.419646
Equation: (x0 - p(x0)) - (x5 * x5)

Index 2, score=0.343521
Equation: -9.518542 - p(x5)



## 5.8.3 Model 3

In [144]:
model_3f = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.176e+01  1.594e+01  y = -11.904
3           6.118e+01  4.750e-03  y = x₂ + -12.056
4           4.339e+01  3.435e-01  y = -9.5175 - p(x₅)
5           8.516e+00  1.628e+00  y = -3.7578 - (x₅ * x₅)
7           8.495e+00  1.253e-03  y = -3.9175 - (x₅ * (x₅ * 0.98039))
9           7.833e+00  4.057e-02  y = sin(exp(x₀)) + (-3.6056 - (x₅ * x₅))
10          4.064e+00  6.561e-01  y = (x₀ * sin(x₀)) + (-3.0806 - (x₅ * x₅))
12          4.005e+00  7.406e-03  y = (-3.0804 - (x₅ * x₅)) + ((x₀ - 0.33452) * sin(x₀))
13          3.818e+00  4.779e-02  y = ((x₅ * -0.98831) * x₅) + ((x₀ - safe_pow(1.6557, x₀)) ...
                                      * 1.3866)
15          3.788e+00  3.946e-03  y = (((x₅ * x₅) * -0.97736) + ((x₀ - safe_pow(1.6583, x₀))...
                                       * 1.3001)) + -0.38597
────────────────────────────────────

PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                         -11.903738   
	1        0.004750                                     x2 + -12.05577   
	2        0.343521                                  -9.517497 - p(x5)   
	3        1.628317                             -3.7577972 - (x5 * x5)   
	4        0.001253                -3.917497 - (x5 * (x5 * 0.9803926))   
	5        0.040574             sin(exp(x0)) + (-3.605619 - (x5 * x5))   
	6  >>>>  0.656053          (x0 * sin(x0)) + (-3.0805976 - (x5 * x5))   
	7        0.007406  (-3.0804355 - (x5 * x5)) + ((x0 - 0.3345231) *...   
	8        0.047786  ((x5 * -0.9883135) * x5) + ((x0 - safe_pow(1.6...   
	9        0.003946  (((x5 * x5) * -0.9773597) + ((x0 - safe_pow(1....   
	
	        loss  complexity  
	0  61.764057           1  
	1  61.180126           3  
	2  43.393124           4  
	3   8.516313           5  
	4   8.494999           7  
	5   7.832882           9  
	6   4.064444          10  
	7   4.004684          12  
	8   3.817818          13  
	9   3.787807          15  
]

  - outputs\20250511_222202_XUS40O\hall_of_fame.csv


In [145]:
best_equation_3 = model_3f.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3f.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x0*sin(x0) - x5*x5 - 3.0805976
Index 3, score=1.628317
Equation: -3.7577972 - (x5 * x5)

Index 6, score=0.656053
Equation: (x0 * sin(x0)) + (-3.0805976 - (x5 * x5))

Index 2, score=0.343521
Equation: -9.517497 - p(x5)



## 5.8.4 Model 1 with noise

In [146]:
model_1f_noisy.fit(X, y_noisy)
best_equation_1_noisy = model_1f_noisy.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1f_noisy.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.161e+01  1.594e+01  y = -11.838
3           6.111e+01  4.091e-03  y = x₂ + -11.99
4           5.750e+01  6.084e-02  y = cos(x₅) + -11.667
5           1.533e+01  1.322e+00  y = (x₅ * x₅) * -1.2373
7           8.531e+00  2.932e-01  y = (x₅ * (x₅ * -0.97867)) + -3.8661
9           8.509e+00  1.294e-03  y = ((x₅ * (x₅ + 0.054393)) * -0.97485) + -3.8899
10          7.496e+00  1.268e-01  y = ((x₅ * (x₅ * -0.98576)) + -3.5404) + cos(x₀)
11          2.886e+00  9.543e-01  y = (x₅ * (x₅ * -0.99999)) + ((x₀ * x₀) * -0.36527)
13          2.613e+00  4.981e-02  y = ((x₅ * (x₅ * -0.96169)) + -0.99184) + ((x₀ * x₀) * -0....
                                      32209)
18          2.586e+00  2.098e-03  y = (x₅ * (x₅ * -0.96062)) + ((exp(x₀ * -0.055306) * -0.98...
                                      384) + (x₀ * (x₀ * -0.32065)))
─────────────────

## 5.8.5 Model 2 with noise

In [147]:
model_2f_noisy.fit(X, y_noisy)
best_equation_2_noisy = model_2f_noisy.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2f_noisy.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.161e+01  1.594e+01  y = -11.839
3           6.109e+01  4.287e-03  y = -11.7 - x₅
4           4.305e+01  3.500e-01  y = -9.4531 - p(x₅)
5           8.557e+00  1.616e+00  y = -3.6923 - (x₅ * x₅)
7           8.532e+00  1.429e-03  y = -3.8312 - safe_pow(x₅ * x₅, 0.99324)
8           7.538e+00  1.239e-01  y = -3.5993 - ((x₅ * x₅) - cos(x₀))
10          5.720e+00  1.379e-01  y = -3.6774 - ((x₅ * x₅) - cos(x₀ * -0.62993))
11          3.489e+00  4.945e-01  y = -0.73883 - ((x₅ * x₅) + safe_pow(safe_pow(1.0943, x₀),...
                                       x₀))
12          2.908e+00  1.820e-01  y = cos(x₀) - ((x₅ * x₅) + safe_pow(safe_pow(1.0971, x₀), ...
                                      x₀))
14          2.732e+00  3.130e-02  y = cos(x₀) - ((safe_pow(safe_pow(1.0925, x₀), x₀) + 0.532...
                                      76) + (x₅ *

## 5.8.6 Model 3 with noise

In [148]:
model_3f_noisy.fit(X, y_noisy)
best_equation_3_noisy = model_3f_noisy.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3f_noisy.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.161e+01  1.594e+01  y = -11.838
3           6.109e+01  4.286e-03  y = -11.701 - x₅
4           4.305e+01  3.500e-01  y = -9.453 - p(x₅)
5           8.557e+00  1.616e+00  y = -3.6927 - (x₅ * x₅)
7           8.531e+00  1.476e-03  y = -3.8661 - ((x₅ * x₅) * 0.97867)
8           5.579e+00  4.248e-01  y = x₀ - ((x₅ * x₅) + p(x₀))
9           2.886e+00  6.589e-01  y = (x₀ * (x₀ * -0.36527)) - (x₅ * x₅)
11          2.694e+00  3.453e-02  y = (x₀ * (x₀ * -0.32014)) - ((x₅ * x₅) + 0.69801)
13          2.613e+00  1.528e-02  y = ((x₀ * x₀) * -0.32211) - (((x₅ * x₅) - -1.0311) * 0.96...
                                      17)
15          2.589e+00  4.642e-03  y = (x₀ * (x₀ * -0.32216)) - ((((x₅ + 0.05796) * x₅) - -1....
                                      061) * 0.95769)
──────────────────────────────────────────────────────────────────────

# 5.9 Output into a DF [5.0: 3.0]

In [149]:
import pandas as pd

data = []

for model_name in ['model_1f', 'model_2f', 'model_3f','model_1f_noisy', 'model_2f_noisy', 'model_3f_noisy']:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1f,best,x0*sin(x0) + x5*x5*(-0.9596014) - 3.1420352,1.314799
1,model_1f,3,(x5 * x5) * -1.2424526,1.314799
2,model_1f,8,(((x5 * x5) * -0.9596014) + (x0 * sin(x0))) + ...,0.615845
3,model_1f,4,((x5 * -0.9803854) * x5) + -3.917589,0.300026
4,model_2f,best,(x0 - sympy_p(x0 - 1*(-0.4466726)))*1.2834208 ...,1.628317
5,model_2f,3,-3.757748 - (x5 * x5),1.628317
6,model_2f,5,(x0 - p(x0)) - (x5 * x5),0.419646
7,model_2f,2,-9.518542 - p(x5),0.343521
8,model_3f,best,x0*sin(x0) - x5*x5 - 3.0805976,1.628317
9,model_3f,3,-3.7577972 - (x5 * x5),1.628317


In [150]:
df.to_csv('results5_for3.csv')

# 5.10 Function f [5.0: 4.0]

$$f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$$

- X (random values  $\mathbb{R}^6$ [-15, 15])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev1 = 2, std_dev2 = 5) 

In [151]:
from numpy import sin, cos, exp, log, sqrt
# Dataset

np.random.seed(0)
X = np.random.uniform(-15, 15, size=(200, 6))
# y = 2.5382 * np.cos(X[:, 3]) + X[:, 0] ** 2 - 2
from sympy import primerange
primes_list = [0] + list(primerange(1, 1000))
primes_dict = {i: primes_list[i] if i < len(primes_list) else 0 for i in range(1000)}

y = [
    2.2 * sin(X[i, 0] + 2 * X[i, 1]) - X[i, 5] ** 2 - primes_dict[abs(int(X[i, 0]))]
    for i in range(200)
]
# Add the y samples with random noise
noise1 = np.random.normal(loc=0.0, scale=2, size=len(y))
y_noisy1 = np.array(y) + noise1
noise2 = np.random.normal(loc=0.0, scale=5, size=len(y))
y_noisy2 = np.array(y) + noise2

In [152]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.11 Experiments [5.0]



## 5.11.1 Model 1

In [153]:
model_1f_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.740e+03  1.594e+01  y = -94.018
3           4.584e+03  1.665e-02  y = x₂ + -94.473
4           4.569e+03  3.288e-03  y = p(x₂) + -98.392
5           3.869e+02  2.469e+00  y = (x₅ * -1.1565) * x₅
7           1.995e+02  3.312e-01  y = (x₅ * (x₅ * -1.0056)) + -20.293
9           1.995e+02  3.952e-05  y = (x₅ * ((x₅ * -1.006) + 0.014937)) + -20.272
10          1.977e+02  9.030e-03  y = (cos(x₀) + ((x₅ * x₅) + 20.221)) * -1.0048
11          1.970e+02  3.316e-03  y = ((x₅ * x₅) + (cos(p(x₁)) + 19.534)) * -1.0058
12          1.576e+02  2.232e-01  y = ((exp(x₀) * -1.5546e-05) + -18.783) + ((x₅ * x₅) * -1....
                                      0104)
15          1.567e+02  1.900e-03  y = (((x₅ * x₅) * -1.0104) + (exp(x₀) * -1.5546e-05)) + (-...
                                      18.783 + sin(1.3475))
17          1.565e+02  7.945e-04  y

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          -94.01778   
	1         0.016655                                     x2 + -94.47284   
	2         0.003288                                 p(x2) + -98.392105   
	3         2.468880                             (x5 * -1.1564522) * x5   
	4   >>>>  0.331178              (x5 * (x5 * -1.0056486)) + -20.292524   
	5         0.000040  (x5 * ((x5 * -1.0060071) + 0.014937467)) + -20...   
	6         0.009030   (cos(x0) + ((x5 * x5) + 20.220713)) * -1.0047513   
	7         0.003316  ((x5 * x5) + (cos(p(x1)) + 19.534245)) * -1.00...   
	8         0.223236  ((exp(x0) * -1.5546042e-5) + -18.782547) + ((x...   
	9         0.001900  (((x5 * x5) * -1.0103766) + (exp(x0) * -1.5546...   
	10        0.000795  (x5 * (x5 * -1.0103766)) + (((exp(x0) * -1.554...   
	11        0.001838  (sin(exp(x0) + -18.782547) + ((exp(x0) * -1.55...   
	12        0.004475  ((x5 * (x5 * -1.0103766)) + (((exp(x0) * -1.55...   
	
	          loss  complexity  
	0   4739.50630           1  
	1   4584.23730           3  
	2   4569.19000           4  
	3    386.91740           5  
	4    199.50821           7  
	5    199.49242           9  
	6    197.69907          10  
	7    197.04459          11  
	8    157.62108          12  
	9    156.72510          15  
	10   156.47623          17  
	11   156.18884          18  
	12   155.49146          19  
]

### Model 1 best + top3 according to score

In [154]:
best_equation_1 = model_1f_wide_range.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1f_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x5*x5*(-1.0056486) - 20.292524
Index 3, score=2.468880
Equation: (x5 * -1.1564522) * x5

Index 4, score=0.331178
Equation: (x5 * (x5 * -1.0056486)) + -20.292524

Index 8, score=0.223236
Equation: ((exp(x0) * -1.5546042e-5) + -18.782547) + ((x5 * x5) * -1.0103766)



## 5.11.2 Model 2

Added constraint to operator "^" - safe_pow

In [155]:
def safe_pow(x, y):
    if x < 0 and np.floor(y) != y:
        return np.nan
    return np.power(x, y)

model_2f_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.740e+03  1.594e+01  y = -94.018
3           4.584e+03  1.665e-02  y = x₂ + -94.472
4           3.698e+03  2.149e-01  y = -83.176 - p(x₅)
5           1.997e+02  2.919e+00  y = -20.705 - (x₅ * x₅)
7           1.995e+02  3.588e-04  y = ((x₅ * x₅) - -20.178) * -1.0056
8           1.667e+02  1.797e-01  y = (x₀ - p(x₀)) - (x₅ * x₅)
10          6.971e+01  4.359e-01  y = (-9.8479 - (x₅ * x₅)) + (x₀ - p(x₀))
12          5.193e+01  1.473e-01  y = (-9.4468 - (x₅ * x₅)) - (p(x₀) + (x₀ * -1.4617))
13          3.942e+00  2.578e+00  y = (0.92209 - (x₅ * x₅)) - (p(x₀) + p(x₀ * -0.96002))
14          3.941e+00  2.652e-05  y = (cos(0.45248) - (x₅ * x₅)) - (p(x₀) + p(x₀ * -0.96002)...
                                      )
15          2.030e+00  6.635e-01  y = (x₅ * (0.014751 - x₅)) - (p(x₀ - 0.52331) + p(-0.50788...
                                

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          -94.01777   
	1         0.016655                                    x2 + -94.472084   
	2         0.214858                                 -83.176254 - p(x5)   
	3         2.918948                             -20.704887 - (x5 * x5)   
	4         0.000359               ((x5 * x5) - -20.17797) * -1.0056484   
	5         0.179696                           (x0 - p(x0)) - (x5 * x5)   
	6         0.435887              (-9.84791 - (x5 * x5)) + (x0 - p(x0))   
	7         0.147264  (-9.446785 - (x5 * x5)) - (p(x0) + (x0 * -1.46...   
	8         2.578274  (0.9220923 - (x5 * x5)) - (p(x0) + p(x0 * -0.9...   
	9         0.000027  (cos(0.45248446) - (x5 * x5)) - (p(x0) + p(x0 ...   
	10  >>>>  0.663472  (x5 * (0.014750904 - x5)) - (p(x0 - 0.52331203...   
	11        0.000990  ((0.022340596 - x5) * x5) - ((p(-0.5079461 - x...   
	12        0.000086  (x5 * (p(0.8790137 - ((x5 + 0.8790137) * x5)) ...   
	13        0.009250  ((p(0.6209776 - ((x5 + cos(0.24246866)) * x5))...   
	14        0.000606  ((p(0.6209776 - (x5 * (cos(p(safe_pow(0.879013...   
	
	           loss  complexity  
	0   4739.506300           1  
	1   4584.237300           3  
	2   3697.901400           4  
	3    199.651460           5  
	4    199.508210           7  
	5    166.694000           8  
	6     69.712970          10  
	7     51.927998          12  
	8      3.941591          13  
	9      3.941486          14  
	10     2.030101          15  
	11     2.026084          17  
	12     2.025218          22  
	13     2.006570          23  
	14     2.000498          28  
]

  - outputs\20250511_222217_O0LUpQ\hall_of_fame.csv


In [156]:
best_equation_2 = model_2f_wide_range.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2f_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x5*(0.014750904 - x5) - (sympy_p(-x0 - 0.50787765) + sympy_p(x0 - 1*0.52331203))
Index 3, score=2.918948
Equation: -20.704887 - (x5 * x5)

Index 8, score=2.578274
Equation: (0.9220923 - (x5 * x5)) - (p(x0) + p(x0 * -0.96002203))

Index 10, score=0.663472
Equation: (x5 * (0.014750904 - x5)) - (p(x0 - 0.52331203) + p(-0.50787765 - x0))



## 5.11.3 Model 3

In [157]:
model_3f_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.740e+03  1.594e+01  y = -94.018
3           4.584e+03  1.665e-02  y = x₂ + -94.472
4           3.698e+03  2.149e-01  y = -83.179 - p(x₅)
5           1.997e+02  2.919e+00  y = -20.707 - (x₅ * x₅)
7           1.995e+02  3.588e-04  y = (x₅ * (x₅ * -1.0057)) + -20.292
8           1.667e+02  1.797e-01  y = x₀ - ((x₅ * x₅) + p(x₀))
9           1.513e+02  9.698e-02  y = -16.069 - ((x₅ * x₅) + safe_pow(1.2577, x₀))
10          6.971e+01  7.748e-01  y = x₀ - ((x₅ * x₅) + (p(x₀) + 9.8483))
12          3.887e+01  2.921e-01  y = (x₀ * 2.3441) - (p(x₀ + 4.4128) + (x₅ * x₅))
14          3.815e+01  9.351e-03  y = ((x₀ + -0.36427) * 2.3277) - (p(x₀ + 4.4078) + (x₅ * x...
                                      ₅))
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_222223_kHuwii\ha

PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                          -94.01778   
	1        0.016655                                     x2 + -94.47222   
	2        0.214858                                  -83.17851 - p(x5)   
	3        2.918948                             -20.706514 - (x5 * x5)   
	4        0.000359               (x5 * (x5 * -1.0056516)) + -20.29247   
	5        0.179696                           x0 - ((x5 * x5) + p(x0))   
	6        0.096979   -16.06928 - ((x5 * x5) + safe_pow(1.257658, x0))   
	7        0.774795              x0 - ((x5 * x5) + (p(x0) + 9.848336))   
	8  >>>>  0.292071  (x0 * 2.3441305) - (p(x0 + 4.4128265) + (x5 * ...   
	9        0.009351  ((x0 + -0.36426887) * 2.3277235) - (p(x0 + 4.4...   
	
	          loss  complexity  
	0  4739.506300           1  
	1  4584.237000           3  
	2  3697.901400           4  
	3   199.651460           5  
	4   199.508210           7  
	5   166.694000           8  
	6   151.287340           9  
	7    69.712970          10  
	8    38.870853          12  
	9    38.150658          14  
]

In [158]:
best_equation_3 = model_3f_wide_range.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3f_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x0*2.3441305 - (x5*x5 + sympy_p(x0 + 4.4128265))
Index 3, score=2.918948
Equation: -20.706514 - (x5 * x5)

Index 7, score=0.774795
Equation: x0 - ((x5 * x5) + (p(x0) + 9.848336))

Index 8, score=0.292071
Equation: (x0 * 2.3441305) - (p(x0 + 4.4128265) + (x5 * x5))



## 5.11.4 Model 1 with noise std_dev 2 and 5

In [159]:
model_1f_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_1_noisy = model_1f_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1f_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.733e+03  1.594e+01  y = -93.755
3           4.579e+03  1.657e-02  y = x₂ + -94.213
4           4.527e+03  1.127e-02  y = p(x₂) + -104.17
5           3.836e+02  2.468e+00  y = (x₅ * -1.1542) * x₅
7           2.000e+02  3.257e-01  y = (x₅ * (x₅ * -1.0049)) + -20.087
9           2.000e+02  1.788e-07  y = (((x₅ + -0.0013122) * -1.0049) * x₅) + -20.085
10          1.048e+02  6.459e-01  y = (-1.07 * ((x₅ * x₅) + p(x₀))) + x₀
12          7.177e+01  1.894e-01  y = (((p(x₀) + (x₅ * x₅)) * -1.006) + -9.088) + x₀
13          8.040e+00  2.189e+00  y = ((p(x₀) + p(x₀ * -0.95892)) + (x₅ * x₅)) * -0.99201
15          5.877e+00  1.567e-01  y = ((p(x₀ * -0.95871) + (x₅ * x₅)) + p(x₀ + -0.50896)) * ...
                                      -0.9922
───────────────────────────────────────────────────────────────────────────────────────────────────
  -

In [160]:
model_1f_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_1_noisy = model_1f_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1f_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.816e+03  1.594e+01  y = -93.859
3           4.654e+03  1.718e-02  y = x₂ + -94.32
5           4.151e+02  1.208e+00  y = (x₅ * -1.1573) * x₅
7           2.365e+02  2.814e-01  y = (x₅ * (x₅ * -1.0101)) + -19.813
9           2.364e+02  1.642e-04  y = (x₅ * ((x₅ + 0.032859) * -1.0093)) + -19.857
10          2.347e+02  7.258e-03  y = ((sin(x₅) + (x₅ * x₅)) * -1.0093) + -19.925
11          3.355e+01  1.945e+00  y = (x₀ * (x₀ * -0.22471)) + ((x₅ * x₅) * -1.0113)
13          3.123e+01  3.582e-02  y = (((x₀ * (x₀ * 0.21096)) + (x₅ * x₅)) * -0.99897) + -2....
                                      8867
14          2.599e+01  1.838e-01  y = ((x₅ * x₅) * -1.0025) + ((cos(x₀ * 0.16475) * 25.139) ...
                                      + -24.603)
17          2.533e+01  8.484e-03  y = cos(x₄) + ((x₅ * (x₅ * -1.0026)) + ((cos(x₀ * 0.16661)...
   

## 5.11.5 Model 2 with noise std_dev 2 and 5

In [161]:
model_2f_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_2_noisy = model_2f_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2f_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.733e+03  1.594e+01  y = -93.757
3           4.579e+03  1.657e-02  y = x₂ + -94.213
4           3.689e+03  2.160e-01  y = -82.916 - p(x₅)
5           2.001e+02  2.915e+00  y = -20.445 - (x₅ * x₅)
7           2.000e+02  2.677e-04  y = -20.086 - (x₅ * (x₅ * 1.0049))
8           1.638e+02  1.993e-01  y = x₀ - ((x₅ * x₅) + p(x₀))
9           1.417e+01  2.448e+00  y = ((x₀ * -0.227) * x₀) - (x₅ * x₅)
11          9.652e+00  1.919e-01  y = ((x₀ * x₀) * -0.20273) + (-3.3798 - (x₅ * x₅))
13          9.551e+00  5.242e-03  y = ((x₀ * ((x₀ + -0.17139) * -0.20272)) + -3.3503) - (x₅ ...
                                      * x₅)
15          9.499e+00  2.701e-03  y = ((((x₀ * x₀) * -0.20423) + ((-0.0034036 - x₅) * x₅)) +...
                                       -3.8077) * 0.99411
17          7.170e+00  1.407e-01  y = ((x₀ * (x₀ * ((x₀ * (x₀ * 0.

In [162]:
model_2f_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_2_noisy = model_2f_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2f_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.816e+03  1.594e+01  y = -93.867
3           4.654e+03  1.718e-02  y = x₂ - 94.319
5           2.369e+02  1.489e+00  y = -20.551 - (x₅ * x₅)
7           2.365e+02  9.611e-04  y = ((x₅ * x₅) * -1.0101) - 19.813
8           2.352e+02  5.440e-03  y = -20.943 - ((x₅ * x₅) + sin(x₅))
10          2.347e+02  1.048e-03  y = (x₅ * (x₅ * -1.0092)) - (sin(x₅) + 19.934)
11          2.315e+02  1.373e-02  y = ((x₅ * x₅) * (-1.0066 - (x₀ * -0.002409))) + -19.842
12          1.729e+02  2.920e-01  y = (x₅ * (x₅ * -1.0044)) - (exp(x₀ * -0.2289) - -13.684)
14          1.057e+02  2.460e-01  y = (x₅ * (x₅ * -1.0122)) - (exp(x₀ * 0.27158) - (x₀ + -11...
                                      .227))
16          6.986e+01  2.070e-01  y = (0.41387 - (x₅ * x₅)) - (exp((x₀ * 0.14272) + 2.4535) ...
                                      - (x₀ * 2.6555))
18      

## 5.11.6 Model 3 with noise

In [163]:
model_3f_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_3_noisy = model_3f_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3f_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.733e+03  1.594e+01  y = -93.756
3           4.579e+03  1.657e-02  y = x₂ + -94.213
4           3.689e+03  2.160e-01  y = -82.916 - p(x₅)
5           2.001e+02  2.915e+00  y = -20.445 - (x₅ * x₅)
7           2.000e+02  2.677e-04  y = (-19.989 - (x₅ * x₅)) * 1.0049
8           1.980e+02  9.916e-03  y = -20.503 - ((x₅ * x₅) + sin(x₅))
9           1.834e+02  7.659e-02  y = -20.503 - ((x₅ * x₅) + safe_pow(0.86783, x₀))
11          1.417e+01  1.280e+00  y = ((-0.00089709 - x₅) * x₅) + ((x₀ * x₀) * -0.227)
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_222238_tULtv1\hall_of_fame.csv
Best by score: x0*x0*(-0.22699876) + x5*(-x5 - 0.00089708576)
Index 3, score=2.914507
Equation: -20.444698 - (x5 * x5)

Index 7, score=1.280336
Equation: ((-0.00089708576 - x5) * x5) + (

In [164]:
model_3f_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_3_noisy = model_3f_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3f_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.816e+03  1.594e+01  y = -93.862
3           4.654e+03  1.718e-02  y = x₂ + -94.319
4           3.757e+03  2.140e-01  y = -83.023 - p(x₅)
5           2.369e+02  2.764e+00  y = -20.524 - safe_pow(x₅, 2)
8           1.993e+02  5.762e-02  y = (x₀ - safe_pow(x₅, 2)) - p(x₀)
10          1.116e+02  2.899e-01  y = ((x₀ - safe_pow(x₅, 2)) - p(x₀)) - 7.194
12          3.817e+01  5.365e-01  y = ((x₀ * 3) - safe_pow(x₅, 2)) - (p(x₀) * 2)
14          3.402e+01  5.744e-02  y = ((2 - safe_pow(x₅, 2)) - (p(x₀) * 2)) + (x₀ * 3)
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_222240_EFXl2L\hall_of_fame.csv
Best by score: x0*3.0 - x5**2.0 - 2.0*sympy_p(x0)
Index 3, score=2.763808
Equation: -20.523565 - safe_pow(x5, 2.0)

Index 6, score=0.536518
Equation: ((x0 * 3.0) - safe_pow(x

# 5.12 Output into a DF [5.0]

In [165]:
import pandas as pd

data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5',
    'model_1f', 'model_2f', 'model_3f',
    'model_1f_noisy', 'model_2f_noisy', 'model_3f_noisy',
    'model_1f_wide_range', 'model_2f_wide_range', 'model_3f_wide_range',
    'model_1f_noisy_std_dev_2', 'model_2f_noisy_std_dev_2', 'model_3f_noisy_std_dev_2',
    'model_1f_noisy_std_dev_5', 'model_2f_noisy_std_dev_5', 'model_3f_noisy_std_dev_5',    
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,(x5*x5 + 3.2226162)*(-0.98154366),2.031154
1,model_1,3,(x5 * x5) * -1.1931033,2.031154
2,model_1,4,((x5 * x5) + 3.2226162) * -0.98154366,0.560019
3,model_1,11,((x5 * x5) + (cos(cos(x1 * 2.0104563) + sin(x0...,0.183691
4,model_2,best,-x5*x5 - 3.012808,3.142667
...,...,...,...,...
115,model_2f_noisy_std_dev_5,7,(x5 * (x5 * -1.0044272)) - (exp(x0 * -0.228904...,0.292030
116,model_3f_noisy_std_dev_5,best,x0*3.0 - x5**2.0 - 2.0*sympy_p(x0),2.763808
117,model_3f_noisy_std_dev_5,3,"-20.523565 - safe_pow(x5, 2.0)",2.763808
118,model_3f_noisy_std_dev_5,6,"((x0 * 3.0) - safe_pow(x5, 2.0)) - (p(x0) * 2.0)",0.536518


In [166]:
df.to_csv('results5.csv')

In [167]:
data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5',
    'model_1f', 'model_2f', 'model_3f',
    'model_1f_noisy', 'model_2f_noisy', 'model_3f_noisy',
    'model_1f_wide_range', 'model_2f_wide_range', 'model_3f_wide_range',
    'model_1f_noisy_std_dev_2', 'model_2f_noisy_std_dev_2', 'model_3f_noisy_std_dev_2',
    'model_1f_noisy_std_dev_5', 'model_2f_noisy_std_dev_5', 'model_3f_noisy_std_dev_5',    
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,(x5*x5 + 3.2226162)*(-0.98154366),2.031154
1,model_2,best,-x5*x5 - 3.012808,3.142667
2,model_3,best,-x5*x5 - 3.0128005,3.201109
3,model_1_noisy,best,x5*(-0.9861936)*x5 + sin(x1*(-2.0073497) + sin...,2.018258
4,model_2_noisy,best,-x5*x5 - cos(sin(sin((x1 - sin(x0))*1.9730418)...,3.038596
5,model_3_noisy,best,-x5*x5 - 2.9476075,3.096697
6,model_1_wide_range,best,x5*x5*(-0.9992922) - 3.028511,3.273026
7,model_2_wide_range,best,-x5*x5 - 2.9765701,3.827115
8,model_3_wide_range,best,-x5*x5 - 2.976615,3.827114
9,model_1_noisy_std_dev_2,best,x5*(-0.99853265)*x5 - 2.821686,3.074411


In [168]:
df.to_csv('results5_only_best.csv')